In [44]:
import pandas as pd
from sqlalchemy import create_engine, text
from dotenv import dotenv_values

In [45]:
env = dotenv_values(".env")

In [46]:
engine = create_engine(env["POSTGRES_URI"], pool_pre_ping=True, pool_recycle=900, pool_timeout=30)


def check_postgres_connection():
    try:
        with engine.connect() as connection:
            result = connection.execute(text("SELECT 'Connected Successfully'"))
            return result.fetchone()[0]
    except Exception as e:
        return f"Error connecting to PostgreSQL: {e}"

check_postgres_connection()

'Connected Successfully'

In [47]:
display(pd.read_sql("SELECT COUNT(*) from dataengineering_project_customers", engine))
display(pd.read_sql("SELECT COUNT(*) from dataengineering_project_orders", engine))

,count
0,2000


,count
0,200000


In [48]:
customers_df = pd.read_sql("SELECT * from dataengineering_project_customers", engine)

In [49]:
customers_df.head(5)

,customer_id,name,city,segment,signup_date
0,1,Usman Sheikh,Karachi,Consumer,2022-11-28
1,2,Iqra Khan,Faisalabad,Consumer,2022-11-11
2,3,Maryam Raza,Faisalabad,Consumer,2023-10-20
3,4,Ali Sheikh,Lahore,Consumer,2022-08-09
4,5,Hamza Qureshi,Karachi,Consumer,2022-11-10


In [50]:
display(customers_df.shape)
print("\n", "--"*50, "\n")
display(customers_df.columns)
print("\n", "--"*50, "\n")
display(customers_df.info())
print("\n", "--"*50, "\n")
display(customers_df.describe())

(2000, 5)


 ---------------------------------------------------------------------------------------------------- 



Index(['customer_id', 'name', 'city', 'segment', 'signup_date'], dtype='str')


 ---------------------------------------------------------------------------------------------------- 

<class 'pandas.DataFrame'>
RangeIndex: 2000 entries, 0 to 1999
Data columns (total 5 columns):
 #   Column       Non-Null Count  Dtype
---  ------       --------------  -----
 0   customer_id  2000 non-null   int64
 1   name         2000 non-null   str  
 2   city         2000 non-null   str  
 3   segment      2000 non-null   str  
 4   signup_date  2000 non-null   str  
dtypes: int64(1), str(4)
memory usage: 78.3 KB


None


 ---------------------------------------------------------------------------------------------------- 



,customer_id
count,2000.000000
mean,1000.500000
std,577.494589
min,1.000000
25%,500.750000
50%,1000.500000
75%,1500.250000
max,2000.000000


In [51]:
orders_df = []

In [52]:
orders_counts = pd.read_sql("SELECT COUNT(*) from dataengineering_project_orders", engine)
print(orders_counts)
# orders_df = pd.read_sql("SELECT * from dataengineering_project_orders", engine)

    count
0  200000


In [53]:
from tqdm.auto import tqdm

orders, rows = [], 0

for chunk in tqdm(
    pd.read_sql(
        "SELECT * FROM dataengineering_project_orders",
        engine,
        chunksize=50_000
    ),
    desc="Loading Orders",
    unit="chunk"
):
    rows += len(chunk)
    orders.extend(chunk.to_dict("records"))

print(f"Total rows loaded: {rows:,}")

Loading Orders: 4chunk [00:09,  2.28s/chunk]

Total rows loaded: 200,000


In [54]:
len(orders)

200000

In [62]:
import certifi
from pymongo import MongoClient

try:
    client = MongoClient( env["MONGO_URI"], serverSelectionTimeoutMS=5000, tlsCAFile=certifi.where() )
    
    client.admin.command('ping')
    print("Successfully connected to MongoDB!")

    db = client["dataengineering_db"]
    orders_df = pd.DataFrame(list(db["orders"].find({})))

    print(orders_df.head())
    print(f"Shape: {orders_df.shape}")

except Exception as e:
    print(f"Connection failed: {e}")

Successfully connected to MongoDB!
                        _id  order_id           order_date     status  \
0  6ae8c4a71dc54dd2c64a6131    238188  2025-01-01T02:57:21    shipped   
1  6ae8c4a71dc54dd2c64a6132    110570  2025-01-01T05:06:31  delivered   
2  6ae8c4a71dc54dd2c64a6133    237480  2025-01-01T05:14:21    pending   
3  6ae8c4a71dc54dd2c64a6134    225926  2025-01-01T10:09:15    pending   
4  6ae8c4a71dc54dd2c64a6135    267865  2025-01-01T13:59:25  delivered   

     amount                                         items  \
0   3153.90        [{'category': 'Books', 'quantity': 1}]   
1  30188.21         [{'category': 'Home', 'quantity': 3}]   
2  14310.06      [{'category': 'Fashion', 'quantity': 1}]   
3  55997.47  [{'category': 'Electronics', 'quantity': 1}]   
4  18810.34  [{'category': 'Electronics', 'quantity': 1}]   

                                          payment          updated_at  \
0           {'method': 'card', 'installments': 3} 2025-01-04 05:39:21   
1            